In [17]:
import re

import numpy as np
import pandas as pd

In [18]:
df = pd.read_csv('../data/events/2_struct/2000-2025.csv')

df.head()

,date_start,date_end,event
0,2000-01-01,NaN,Деноминация белорусского рубля;
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н..."
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог..."
3,2000-01-02,NaN,крушение украинского сухогруза типа «река-море...
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5641 entries, 0 to 5640
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date_start  5641 non-null   object
 1   date_end    602 non-null    object
 2   event       5641 non-null   object
dtypes: object(3)
memory usage: 132.3+ KB


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")

ru_stopwords = stopwords.words("russian")


def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^а-яё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


texts = df["event"].dropna().astype(str).apply(clean_text)

vectorizer = TfidfVectorizer(
    max_df=0.9,
    min_df=5,
    stop_words=ru_stopwords,
    ngram_range=(1, 3),
    sublinear_tf=True,
)
X = vectorizer.fit_transform(texts)

n_topics = 10
model = NMF(n_components=n_topics, random_state=42)
W = model.fit_transform(X)
H = model.components_

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic in enumerate(H):
    top_words = [feature_names[j] for j in topic.argsort()[:-11:-1]]
    topics[f"Topic {i + 1}"] = top_words

print(len(vectorizer.vocabulary_))
topics

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ruslan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


3313


{'Topic 1': ['парламентские выборы',
  'парламентские',
  'выборы',
  'досрочные парламентские',
  'досрочные парламентские выборы',
  'досрочные',
  'партия',
  'большинство',
  'сирии',
  'мест'],
 'Topic 2': ['человек',
  'погибли',
  'погибли человек',
  'результате',
  'человека',
  'человек погибли',
  'получили',
  'ранения',
  'ранены',
  'погибло'],
 'Topic 3': ['должность',
  'вступил',
  'должность президента',
  'президента',
  'вступил должность',
  'вступил должность президента',
  'года',
  'президент',
  'должность президент',
  'вступил должность президент'],
 'Topic 4': ['тур',
  'выборов',
  'президентских',
  'президентских выборов',
  'второй',
  'второй тур',
  'тур президентских',
  'тур президентских выборов',
  'второй тур президентских',
  'одержал'],
 'Topic 5': ['мира',
  'чемпионат',
  'чемпионат мира',
  'россия',
  'мира хоккею',
  'хоккею',
  'чемпионат мира хоккею',
  'сборная',
  'шайбой',
  'хоккею шайбой'],
 'Topic 6': ['премьер',
  'министром',
  'п

In [21]:
from sklearn.decomposition import TruncatedSVD

n_topics = 10

lsa = TruncatedSVD(
    n_components=n_topics,
    random_state=42
)

X_lsa = lsa.fit_transform(X)
feature_names = vectorizer.get_feature_names_out()

topics = {}

for i, comp in enumerate(lsa.components_):
    indices = np.argsort(np.abs(comp))[-12:]
    top_words = [feature_names[j] for j in indices]
    topics[f"Topic {i + 1}"] = top_words

topics

{'Topic 1': ['тур',
  'победу одержала',
  'победу одержал',
  'одержал',
  'одержала',
  'партия',
  'президентские выборы',
  'президентские',
  'победу',
  'парламентские выборы',
  'парламентские',
  'выборы'],
 'Topic 2': ['президента',
  'погибло',
  'получили ранения',
  'ранены',
  'ранения',
  'получили',
  'человек погибли',
  'человека',
  'результате',
  'погибли человек',
  'погибли',
  'человек'],
 'Topic 3': ['одержал',
  'погибли',
  'человек',
  'тур',
  'выборов',
  'парламентские',
  'парламентские выборы',
  'вступил должность',
  'должность президента',
  'вступил',
  'президента',
  'должность'],
 'Topic 4': ['одержал',
  'победу',
  'второй',
  'президентских выборов',
  'второй тур',
  'президентских',
  'вступил должность',
  'должность президента',
  'выборов',
  'тур',
  'вступил',
  'должность'],
 'Topic 5': ['шайбой',
  'мира хоккею шайбой',
  'хоккею шайбой',
  'одержала',
  'чемпионат мира хоккею',
  'мира хоккею',
  'хоккею',
  'сборная',
  'россия',
  '

In [22]:
from sklearn.decomposition import LatentDirichletAllocation

n_topics = 10
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method="batch",
    max_iter=30,
    doc_topic_prior=0.1,  # alpha
    topic_word_prior=0.01  # beta
)
X_lda = lda.fit_transform(X)

feature_names = vectorizer.get_feature_names_out()
topics = {}
for i, topic_dist in enumerate(lda.components_):
    top_idx = topic_dist.argsort()[-12:][::-1]
    topics[f"Topic {i + 1}"] = [feature_names[j] for j in top_idx]

topics

{'Topic 1': ['сша',
  'союз',
  'корабля',
  'экипаж',
  'тма',
  'союз тма',
  'космического',
  'космического корабля',
  'корабля союз',
  'владимир',
  'космический',
  'путин'],
 'Topic 2': ['премьер',
  'министром',
  'премьер министром',
  'стал',
  'отставку',
  'министр',
  'премьер министр',
  'новым',
  'президент',
  'новым премьер',
  'новым премьер министром',
  'лидер'],
 'Topic 3': ['космодрома',
  'запуск',
  'открытие',
  'мире',
  'казахстан',
  'официально',
  'саммит',
  'впервые',
  'стала',
  'байконур',
  'космодрома байконур',
  'истории'],
 'Topic 4': ['россии',
  'начало',
  'территории',
  'кндр',
  'первого',
  'сша',
  'лет',
  'стран',
  'сирии',
  'решение',
  'государств',
  'власти'],
 'Topic 5': ['выборы',
  'парламентские',
  'парламентские выборы',
  'победу',
  'президентские',
  'президентские выборы',
  'партия',
  'одержал',
  'победу одержал',
  'президента',
  'тур',
  'выборов'],
 'Topic 6': ['убийство',
  'открыт',
  'нато',
  'должность пре

In [23]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

texts = df["event"]

embedding_model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

vectorizer_model = CountVectorizer(
    ngram_range=(1, 3),
    min_df=5
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts)

2025-12-15 22:21:49,928 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-15 22:22:24,168 - BERTopic - Embedding - Completed ✓
2025-12-15 22:22:24,169 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-15 22:22:25,334 - BERTopic - Dimensionality - Completed ✓
2025-12-15 22:22:25,335 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-15 22:22:26,578 - BERTopic - Cluster - Completed ✓
2025-12-15 22:22:26,582 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-15 22:22:26,826 - BERTopic - Representation - Completed ✓


In [24]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1442,-1_на_по_человек_тур,"[на, по, человек, тур, результате, чемпионат, ...",[Второй тур президентских выборов в Литве. Поб...
1,0,183,0_перу_всеобщие_президента_чили,"[перу, всеобщие, президента, чили, всеобщие вы...",[второй тур президентских выборов в Аргентине....
2,1,145,1_самолёт_борту_на борту_все,"[самолёт, борту, на борту, все, разбился, ката...","[самолёт Ан-148, выполнявший рейс из Москвы в ..."
3,2,137,2_протеста_против_массовые_протесты,"[протеста, против, массовые, протесты, началис...",[в Армении начались протесты против уступок по...
4,3,132,3_армении_казахстане_парламентских выборов_пре...,"[армении, казахстане, парламентских выборов, п...",[Президентские выборы в Азербайджане; победу о...
5,4,96,4_союз_корабля_космического корабля_космического,"[союз, корабля, космического корабля, космичес...",[приземление космического корабля Союз ТМА-19....
6,5,84,5_сша_президент сша_джордж_джо,"[сша, президент сша, джордж, джо, представител...",[Президент США Барак Обама подписал Закон о фи...
7,6,82,6_саммит_государств_конференция_глав,"[саммит, государств, конференция, глав, нато, ...","[саммит G20., саммит АТЭС (Манила, Филиппины)...."
8,7,79,7_произошло_землетрясения_человек_более,"[произошло, землетрясения, человек, более, без...",[В Тбилиси произошло землетрясение магнитудой ...
9,8,72,8_сербии_хорватии_голосов_выборах,"[сербии, хорватии, голосов, выборах, парламент...",[Парламентские выборы в Сербии. По предварител...


In [25]:
# semi-supervised learning
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=True,
    zeroshot_topic_list=[
        'природная катастрофа', 'авиакатастрофа', 'государственный переворот', 'вооруженный конфликт', 'теракт', 'протесты', 'санкции', 'спорт', 'закон'
    ],
    seed_topic_list=[
        ["землетрясение", "цунами", "извержение вулкана", "ураган", "тайфун",
         "наводнение", "оползень", "сель", "засуха", "лесной пожар"],
        ["авиакатастрофа", "крушение самолета", "пассажирский самолет",
         "на борту", "рейс", "экипаж"],
        ["государственный переворот", "госпереворот", "военный переворот",
 "свержение власти", "захват власти", "путч"],
        ["вооруженный конфликт", "война", "военные действия", "наступление", "обстрел", "ВС РФ", "ВСУ"],
        ["теракт", "смертник", "террористический акт"],
        ["санкции", "пакет санкций"],
        ["акция протеста", "протест", "массовые протесты", "беспорядки"],
        ["запуск ракеты", "космос", "спутник", "космический аппарат", "орбита", "космодром"],
        ["чемпионат мира", "спорт", "золотая медаль", "олимпийские игры", "сборная"],
        ["выборы президента", "выборы премьер-министра", "парламентские выборы"],
        ["закон", "подписание закона", "вступление в силу закона", "законопроект", "принятие закона"],
        ["Nvidia", "Microsoft", "Google", "Samsung", "Huawei", "Facebook", "Apple"]
    ]
)

topics, probs = topic_model.fit_transform(texts)

2025-12-15 22:22:27,169 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-15 22:23:01,912 - BERTopic - Embedding - Completed ✓
2025-12-15 22:23:01,913 - BERTopic - Guided - Find embeddings highly related to seeded topics.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-15 22:23:02,026 - BERTopic - Guided - Completed ✓
2025-12-15 22:23:02,027 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-12-15 22:23:03,687 - BERTopic - Dimensionality - Completed ✓
2025-12-15 22:23:03,688 - BERTopic - Zeroshot Step 1 - Finding documents that could be assigned to either one of the zero-shot topics
2025-12-15 22:23:03,732 - BERTopic - Zeroshot Step 1 - Completed ✓
2025-12-15 22:23:05,496 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-12-15 22:23:06,522 - BERTopic - Cluster - Completed ✓
2025-12-15 22:23:06,522 - BERTopic - Zeroshot Step 2 - Combining topics from zero-shot topic modeling with topics from clustering...
2025-12-15 22:23:06,533 - BERTopic - Zeroshot Step 2 - Completed ✓
2025-12-15 22:23:06,534 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-15 22:23:06,736 - BERTopic - Representation - Completed ✓


In [26]:
topic_model.reduce_topics(
    texts,
    nr_topics=15
)

topics, probs = topic_model.transform(texts)
topic_model.get_topic_info()

2025-12-15 22:23:07,019 - BERTopic - Topic reduction - Reducing number of topics
2025-12-15 22:23:07,116 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-12-15 22:23:07,331 - BERTopic - Representation - Completed ✓
2025-12-15 22:23:07,332 - BERTopic - Topic reduction - Reduced number of topics from 107 to 15


Batches:   0%|          | 0/177 [00:00<?, ?it/s]

2025-12-15 22:23:40,922 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


,Topic,Count,Name,Representation,Representative_Docs
0,-1,1704,-1_по_на_выборы_человек,"[по, на, выборы, человек, победу, президента, ...",[Второй тур выборов президента Республики Кипр...
1,0,922,0_выборы_победу_премьер_президента,"[выборы, победу, премьер, президента, выборов,...",[президентские выборы на Коморах (второй тур)....
2,1,736,1_союз_погибли_на_человек,"[союз, погибли, на, человек, сша, станции, чел...",[В результате возгорания поезда-фуникулёра на ...
3,2,629,2_человек_погибли_результате_на,"[человек, погибли, результате, на, более, поги...",[в результате взрыва и пожара на трубопроводе ...
4,3,495,3_россии_война_вторая_рф,"[россии, война, вторая, рф, на, сша, владимир,...",[Вторая чеченская война: Нападение боевиков на...
5,4,382,4_мира_по_саммит_международный,"[мира, по, саммит, международный, россия, каза...","[Чемпионат мира по летнему биатлону (Тюмень, Р..."
6,5,192,5_отставку_европейского_югославии_президент,"[отставку, европейского, югославии, президент,...",[премьер-министр Италии Сильвио Берлускони ушё...
7,6,137,6_президента_президент_чен_вступил,"[президента, президент, чен, вступил, южной, к...",[военный переворот в Нигере. Захвачена резиден...
8,7,115,7_компания_выход_системы_сша,"[компания, выход, системы, сша, китая, мире, н...",[Компания «Microsoft» выпустила операционную с...
9,8,107,8_отношения_израиль_израиля_между,"[отношения, израиль, израиля, между, оон, согл...","[Бахрейн, Саудовская Аравия, Объединённые Араб..."


In [27]:
import random
from collections import defaultdict


def sample_docs_per_topic(texts, topics, n_samples=10, seed=42):
    random.seed(seed)

    topic_to_docs = defaultdict(list)
    for text, topic in zip(texts, topics):
        topic_to_docs[topic].append(text)

    for topic_id, docs in sorted(topic_to_docs.items()):
        if topic_id == -1:
            print(f"topic {topic_id} | total docs: {len(docs)}")
            continue

        print("=" * 80)
        print(f"TOPIC {topic_id} | total docs: {len(docs)}")
        print("=" * 80)

        sampled = random.sample(docs, min(n_samples, len(docs)))
        for i, doc in enumerate(sampled, 1):
            print(f"{i}. {doc}")
        print()


In [28]:
sample_docs_per_topic(texts, topics, n_samples=10)

topic -1 | total docs: 187
TOPIC 0 | total docs: 1163
1. Поль Кагаме избран президентом Руанды.
2. В Магаданской и Курской областях на выборах губернаторов победили Александр Михайлов и Валентин Цветков соответственно; в Калининградской области назначено повторное голосование
3. Вступил в должность президент Египта Абдель Фаттах Ас-Сиси.
4. досрочные парламентские выборы в Чехии. По предварительным данным победу одержала Чешская социал-демократическая партия.
5. выборы президента Кении. Победил вице-премьер страны Ухуру Кениата.
6. президентские выборы в Грузии.
7. Конституционный референдум в Лихтенштейне
8. Коллегия выборщиков утвердила победу Дональда Трампа на президентских выборах в США.
9. Бранко Црвенковский вступил в должность премьер-министра Македонии (до 12 мая 2004 года).
10. Бывший министр обороны Мухаммед ульд аш-Шейх аль-Газуани победил на президентских выборах в Мавритании.

TOPIC 1 | total docs: 677
1. 102-й старт (STS-98) по программе Спейс Шаттл. 23-й полёт шаттла Ат

In [29]:
def save_topics_barchart(topic_model: BERTopic, out_html="topics_barchart.html", top_n_topics=30):
    """
    Сохраняет интерактивный bar chart с размерами/словами тем (BERTopic).
    """
    fig = topic_model.visualize_barchart(top_n_topics=top_n_topics, n_words=10)
    fig.write_html(out_html)
    return fig


def save_documents_scatter(topic_model, texts, topics, out_html="documents_scatter.html"):
    """
    Сохраняет интерактивный scatter документов по темам (BERTopic).
    """
    fig = topic_model.visualize_documents(docs=texts, topics=topics)
    fig.write_html(out_html)
    return fig

save_topics_barchart(topic_model, out_html="topic_plots/topics_barchart.html")
save_documents_scatter(topic_model, texts, topics, out_html="topic_plots/documents_scatter.html")